In [0]:
volume_path = "/Volumes/capstone_project_07/capstone_07_schema/capstone_07_ext_vol"

cdc_path = f"{volume_path}/cdc_data/document_metadata_changes"

In [0]:
from pyspark.sql.functions import *

cdc_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(cdc_path)
)

display(cdc_df)
cdc_df.printSchema()

In [0]:
print("Total CDC records:", cdc_df.count())
print("Total CDC columns:", len(cdc_df.columns))

In [0]:
from pyspark.sql.functions import input_file_name, col

cdc_with_file = cdc_df.withColumn(
    "source_file",
    col("_metadata.file_path")
)

display(
    cdc_with_file
    .groupBy("source_file")
    .count()
    .orderBy("source_file")
)

In [0]:
from pyspark.sql.functions import col, when, trim, sum

null_profile = cdc_df.select(
    [
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in cdc_df.columns
    ]
)

display(null_profile)

In [0]:
duplicate_events = (
    cdc_df
    .groupBy("metadata_event_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_events)

In [0]:
print(
    "Duplicate event ID groups:",
    duplicate_events.count()
)

In [0]:
total_records = cdc_df.count()

unique_records = cdc_df.dropDuplicates().count()

print("Total records:", total_records)
print("Unique records:", unique_records)
print("Exact duplicate rows:", total_records - unique_records)

In [0]:
from pyspark.sql.functions import col, count, countDistinct, struct

duplicate_analysis = (
    cdc_df
    .groupBy("metadata_event_id")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in cdc_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

display(duplicate_analysis)

In [0]:
access_path = (
    f"{volume_path}/event_data/document_access_audit.csv"
)
access_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(access_path)
)

display(access_df)
access_df.printSchema()
print("Total records:", access_df.count())
print("Total columns:", len(access_df.columns))

In [0]:
from pyspark.sql.functions import col, when, trim, sum

access_null_profile = access_df.select(
    [
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in access_df.columns
    ]
)

display(access_null_profile)

In [0]:
from pyspark.sql.functions import count

duplicate_access = (
    access_df
    .groupBy("access_event_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

display(duplicate_access)

print(
    "Duplicate event ID groups:",
    duplicate_access.count()
)

In [0]:
total_access = access_df.count()

unique_access = access_df.dropDuplicates().count()

print("Total records:", total_access)
print("Unique records:", unique_access)

print(
    "Exact duplicate rows:",
    total_access - unique_access
)

In [0]:
from pyspark.sql.functions import countDistinct, struct

access_duplicate_analysis = (
    access_df
    .groupBy("access_event_id")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in access_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

display(access_duplicate_analysis)

In [0]:
display(
    access_df
    .groupBy("action")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
display(
    access_df
    .groupBy("access_result")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
display(
    access_df
    .groupBy("resource_classification")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
access_df.createOrReplaceTempView(
    "access_audit_eda"
)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM access_audit_eda;

In [0]:
%sql
SELECT
    access_event_id,
    COUNT(*) AS record_count
FROM access_audit_eda
GROUP BY access_event_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    access_result,
    COUNT(*) AS total
FROM access_audit_eda
GROUP BY access_result
ORDER BY total DESC;

In [0]:
search_path = (
    f"{volume_path}/event_data/search_query_events.csv"
)
search_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(search_path)
)

display(search_df)
search_df.printSchema()

print("Total records:", search_df.count())
print("Total columns:", len(search_df.columns))

In [0]:
search_null_profile = search_df.select(
    [
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in search_df.columns
    ]
)

display(search_null_profile)

In [0]:
duplicate_search = (
    search_df
    .groupBy("search_event_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_search)

print(
    "Duplicate search event ID groups:",
    duplicate_search.count()
)

In [0]:
total_search = search_df.count()

unique_search = search_df.dropDuplicates().count()

print("Total records:", total_search)
print("Unique records:", unique_search)
print(
    "Exact duplicate rows:",
    total_search - unique_search
)

In [0]:
search_duplicate_analysis = (
    search_df
    .groupBy("search_event_id")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in search_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

display(search_duplicate_analysis)

In [0]:
display(
    search_df
    .groupBy("search_channel")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
display(
    search_df
    .groupBy("clicked_flag")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
from pyspark.sql.functions import col, trim, expr

display(
    search_df
    .select("response_ms")
    .distinct()
    .orderBy("response_ms")
)

In [0]:
invalid_response = search_df.filter(
    col("response_ms").isNotNull() &
    (trim(col("response_ms")) != "") &
    expr("try_cast(trim(response_ms) AS DOUBLE) IS NULL")
)

display(invalid_response)

In [0]:
from pyspark.sql.functions import min, max, avg, count, expr

search_numeric = search_df.withColumn(
    "response_ms_numeric",
    expr("try_cast(trim(response_ms) AS DOUBLE)")
)

search_numeric.select(
    min("response_ms_numeric").alias("minimum_response_ms"),
    max("response_ms_numeric").alias("maximum_response_ms"),
    avg("response_ms_numeric").alias("average_response_ms"),
    count("response_ms_numeric").alias("valid_numeric_records")
).show()

In [0]:
search_numeric.createOrReplaceTempView(
    "search_numeric_eda"
)


In [0]:
display(spark.sql("""SELECT
    MIN(response_ms_numeric) AS minimum_response,
    MAX(response_ms_numeric) AS maximum_response,
    AVG(response_ms_numeric) AS average_response,
    COUNT(response_ms_numeric) AS valid_records,
    COUNT(*) - COUNT(response_ms_numeric) AS invalid_or_missing
FROM search_numeric_eda
"""))

In [0]:
from pyspark.sql.functions import (
    col, trim, when, sum, count,
    countDistinct, struct, expr
)

# 1. Read dataset
category_path = (
    f"{volume_path}/lookup_data/azure_sql_seed/document_category_lookup.csv"
)

category_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(category_path)
)

# 2. Schema and basic information
category_df.printSchema()

print("Total records:", category_df.count())
print("Total columns:", len(category_df.columns))

# 3. NULL and blank profiling
display(
    category_df.select([
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in category_df.columns
    ])
)

# 4. Duplicate category codes
category_duplicates = (
    category_df
    .groupBy("category_code")
    .count()
    .filter(col("count") > 1)
)

display(category_duplicates)

print(
    "Duplicate category code groups:",
    category_duplicates.count()
)

# 5. Exact duplicate rows
total = category_df.count()
unique = category_df.dropDuplicates().count()

print("Exact duplicate rows:", total - unique)

# 6. Investigate duplicate category codes
display(
    category_df
    .groupBy("category_code")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in category_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

# 7. Profile important categorical columns
for c in [
    "retention_class",
    "default_classification",
    "is_searchable",
    "active_flag",
    "content_type",
    "review_frequency"
]:
    print(f"\nDistribution of {c}")
    display(
        category_df
        .groupBy(c)
        .count()
        .orderBy(col("count").desc())
    )

# 8. Validate retention_years
category_numeric = category_df.withColumn(
    "retention_years_numeric",
    expr("try_cast(trim(retention_years) AS INT)")
)

display(
    category_numeric.select(
        count("retention_years_numeric").alias("valid_numeric"),
        (
            count("*") - count("retention_years_numeric")
        ).alias("invalid_or_missing")
    )
)

# 9. Create SQL view
category_df.createOrReplaceTempView(
    "category_lookup_eda"
)

print("Category Lookup EDA completed.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT category_code) AS unique_categories
FROM category_lookup_eda;

In [0]:
%sql
SELECT
    category_code,
    COUNT(*) AS record_count
FROM category_lookup_eda
GROUP BY category_code
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    default_classification,
    COUNT(*) AS total
FROM category_lookup_eda
GROUP BY default_classification;

In [0]:
from pyspark.sql.functions import (
    col, trim, when, sum, count,
    countDistinct, struct, expr, to_date
)

# 1. Read dataset
role_path = (
    f"{volume_path}/lookup_data/azure_sql_seed/role_permission_lookup.csv"
)

role_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(role_path)
)

# 2. Schema and basic information
role_df.printSchema()

print("Total records:", role_df.count())
print("Total columns:", len(role_df.columns))

# 3. NULL and blank profiling
display(
    role_df.select([
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in role_df.columns
    ])
)

# 4. Duplicate role codes
role_duplicates = (
    role_df
    .groupBy("role_code")
    .count()
    .filter(col("count") > 1)
)

display(role_duplicates)

print("Duplicate role code groups:", role_duplicates.count())

# 5. Exact duplicate rows
total = role_df.count()
unique = role_df.dropDuplicates().count()

print("Exact duplicate rows:", total - unique)

# 6. Investigate duplicate role codes
display(
    role_df
    .groupBy("role_code")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in role_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

# 7. Categorical distributions
for c in [
    "resource_classification",
    "permission_level",
    "allowed_action",
    "approval_required",
    "active_flag",
    "policy_owner",
    "policy_version"
]:
    print(f"\nDistribution of {c}")
    display(
        role_df
        .groupBy(c)
        .count()
        .orderBy(col("count").desc())
    )

# 8. Validate effective and expiry dates
display(
    role_df.select(
        count(
            to_date(col("effective_date")).alias("effective_date")
        ).alias("valid_effective_dates"),
        count(
            to_date(col("expiry_date")).alias("expiry_date")
        ).alias("valid_expiry_dates")
    )
)

# 9. Check date ordering
display(
    role_df.filter(
        to_date(col("expiry_date")) < to_date(col("effective_date"))
    )
)

# 10. Create SQL view and validate
role_df.createOrReplaceTempView("role_permission_eda")

display(spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT role_code) AS unique_role_codes
    FROM role_permission_eda
"""))

display(spark.sql("""
    SELECT role_code, COUNT(*) AS record_count
    FROM role_permission_eda
    GROUP BY role_code
    HAVING COUNT(*) > 1
"""))

print("Role Permission EDA completed.")

In [0]:
from pyspark.sql.functions import (
    col, trim, when, sum, count,
    countDistinct, struct, to_date
)

# 1. Read user master
user_path = (
    f"{volume_path}/master_data/azure_sql_seed/user_master.csv"
)

user_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(user_path)
)

# 2. Schema and basic information
user_df.printSchema()

print("Total records:", user_df.count())
print("Total columns:", len(user_df.columns))

# 3. NULL and blank profiling
display(
    user_df.select([
        sum(
            when(
                col(c).isNull() | (trim(col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in user_df.columns
    ])
)

# 4. Duplicate user IDs
user_duplicates = (
    user_df
    .groupBy("user_id")
    .count()
    .filter(col("count") > 1)
)

display(user_duplicates)
print("Duplicate user ID groups:", user_duplicates.count())

# 5. Exact duplicate rows
total = user_df.count()
unique = user_df.dropDuplicates().count()

print("Exact duplicate rows:", total - unique)

# 6. Investigate duplicate user IDs
display(
    user_df
    .groupBy("user_id")
    .agg(
        count("*").alias("record_count"),
        countDistinct(
            struct(*[col(c) for c in user_df.columns])
        ).alias("distinct_record_count")
    )
    .filter(col("record_count") > 1)
)

# 7. Categorical distributions
for c in [
    "department",
    "role_code",
    "employment_status",
    "location",
    "data_access_tier",
    "business_unit",
    "active_flag"
]:
    print(f"\nDistribution of {c}")
    display(
        user_df
        .groupBy(c)
        .count()
        .orderBy(col("count").desc())
    )

# 8. Validate join_date
display(
    user_df.select(
        count(to_date(col("join_date"))).alias("valid_join_dates"),
        (count("*") - count(to_date(col("join_date"))))
        .alias("invalid_or_missing_join_dates")
    )
)

# 9. Check manager references
display(
    user_df.alias("u")
    .join(
        user_df.alias("m"),
        col("u.manager_id") == col("m.user_id"),
        "left"
    )
    .filter(
        col("u.manager_id").isNotNull() &
        (trim(col("u.manager_id")) != "") &
        col("m.user_id").isNull()
    )
    .select("u.user_id", "u.manager_id")
)

# 10. Create SQL view and validate
user_df.createOrReplaceTempView("user_master_eda")

display(spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT user_id) AS unique_users
    FROM user_master_eda
"""))

print("User Master EDA completed.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_users,
    COUNT(DISTINCT user_id) AS unique_users
FROM user_master_eda;

In [0]:
%sql
SELECT
    user_id,
    COUNT(*) AS record_count
FROM user_master_eda
GROUP BY user_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    role_code,
    COUNT(*) AS user_count
FROM user_master_eda
GROUP BY role_code
ORDER BY user_count DESC;